# Pen Plotter — 事前訓練 (Colab)

CASIA-OLHWDBデータで CharEncoder + StyleEncoder + StrokeGenerator を事前訓練する。
ストローク単位の生成アーキテクチャ（各ストロークを個別に自己回帰生成）。

**使い方:**
1. 「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU** を選択
2. Google Drive の `MyDrive/pen_plotter/data/casia_raw/train/` に CASIA `.pot` ファイルを配置
3. 上から順番にセルを実行

In [ ]:
# Google Drive マウント（データとチェックポイントの永続化）
from google.colab import drive
drive.mount('/content/drive')

# リポジトリをクローン（既にあれば最新コードをpull）
import os
if os.path.exists('/content/pen_plotter/.git'):
    !cd /content/pen_plotter && git pull
else:
    !git clone https://github.com/tsaito18/pen_plotter.git /content/pen_plotter
%cd /content/pen_plotter

# 依存パッケージ
!pip install torch torchvision matplotlib numpy -q

## GPU 確認

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("GPUが検出されません。ランタイム設定でGPUを有効化してください。")

## データ準備

KanjiVGデータのダウンロードと、CASIAデータのアップロード。

CASIAデータは Google Drive の `MyDrive/pen_plotter/data/casia_raw/train/` に `.pot` ファイルを配置してください。

In [ ]:
import os

# KanjiVGダウンロード
!python scripts/prepare_kanjivg.py --download

# CASIAデータをDriveからリンク（事前にDriveにアップロードしておく）
DRIVE_CASIA = "/content/drive/MyDrive/pen_plotter/data/casia_raw"
LOCAL_CASIA = "/content/pen_plotter/data/casia_raw"

if os.path.exists(DRIVE_CASIA):
    os.makedirs(os.path.dirname(LOCAL_CASIA), exist_ok=True)
    if not os.path.exists(LOCAL_CASIA):
        os.symlink(DRIVE_CASIA, LOCAL_CASIA)
    print(f"CASIA data linked from Drive")
    pot_count = len([f for f in os.listdir(os.path.join(LOCAL_CASIA, "train"))
                     if f.endswith(".pot")]) if os.path.isdir(os.path.join(LOCAL_CASIA, "train")) else 0
    print(f"  train/*.pot: {pot_count} files")
else:
    print(f"CASIAデータが見つかりません。")
    print(f"Google Drive に {DRIVE_CASIA}/train/*.pot を配置してください。")

## 訓練パラメータ

In [ ]:
import os

# === 訓練パラメータ ===
MAX_SAMPLES = 300000   # CASIAストロークサンプル数 (0=全量)
EPOCHS = 80
BATCH_SIZE = 256       # ストローク単位サンプルは系列長が短いため大きめに設定可能
LEARNING_RATE = 2e-3
NUM_MIXTURES = 20
NUM_WORKERS = 2
USE_AMP = True         # T4以上なら有効化

# チェックポイント保存先（Driveに保存して永続化）
CHECKPOINT_DIR = "/content/drive/MyDrive/pen_plotter/models"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Samples: {MAX_SAMPLES if MAX_SAMPLES > 0 else 'all'}")
print(f"Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, LR: {LEARNING_RATE}")
print(f"Mixtures: {NUM_MIXTURES}, Workers: {NUM_WORKERS}, AMP: {USE_AMP}")
print(f"Checkpoint: {CHECKPOINT_DIR}")

## 訓練実行

In [ ]:
from src.model.pretrain import PretrainConfig, Pretrainer
from pathlib import Path

config = PretrainConfig(
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    num_mixtures=NUM_MIXTURES,
)

trainer = Pretrainer(
    config=config,
    hand_dir=Path("data/strokes"),
    ref_dir=Path("data/strokes"),
    output_dir=Path(CHECKPOINT_DIR),
    device="cuda",
    pot_dir=Path("data/casia_raw/train"),
    max_samples=MAX_SAMPLES,
    num_workers=NUM_WORKERS,
    amp=USE_AMP,
)

history = trainer.train()
print(f"\nTraining complete! Checkpoint saved to {CHECKPOINT_DIR}")

## Loss 可視化

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(history["losses"])
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Pre-training Loss")
plt.grid(True, alpha=0.3)
plt.savefig(f"{CHECKPOINT_DIR}/loss_curve.png", dpi=100, bbox_inches="tight")
plt.show()
print(f"Final loss: {history['losses'][-1]:.4f}")

## ストローク単位生成の確認

各文字の生成ストローク数が参照と一致するか検証する。

In [ ]:
import json
import numpy as np
from src.model.inference import StrokeInference
from src.model.data_utils import strokes_to_deltas_from_arrays

inf = StrokeInference(f"{CHECKPOINT_DIR}/pretrain_checkpoint.pt")

for char in ['あ', 'い', '学', '文', '書']:
    ref_path = f'data/strokes/{char}/{char}_0.json'
    if not os.path.exists(ref_path):
        print(f'{char}: reference not found, skipping')
        continue
    ref_data = json.loads(open(ref_path).read())
    ref_strokes = [np.array([[pt['x'], pt['y']] for pt in s]) for s in ref_data['strokes']]
    style = strokes_to_deltas_from_arrays(ref_strokes).unsqueeze(0)
    result = inf.generate(style, num_steps=100, temperature=0.8, reference_strokes=ref_strokes)
    print(f'{char}: {len(result)} strokes (ref {len(ref_strokes)})')

## プレビュー

訓練結果のサンプル生成。

In [ ]:
import subprocess

result = subprocess.run([
    "python", "scripts/preview_batch.py",
    "--checkpoint", f"{CHECKPOINT_DIR}/pretrain_checkpoint.pt",
    "--ref-dir", "data/strokes",
    "--output", f"{CHECKPOINT_DIR}/preview_batch.png",
    "--temperature", "0.8",
], capture_output=True, text=True)

if result.returncode == 0:
    from IPython.display import Image, display
    display(Image(f"{CHECKPOINT_DIR}/preview_batch.png"))
else:
    print(f"Preview generation failed:\n{result.stderr}")

## チェックポイントをローカルにダウンロード（オプション）

In [ ]:
from google.colab import files
files.download(f"{CHECKPOINT_DIR}/pretrain_checkpoint.pt")